# Conditional Probabilities in MDPs — Usage Examples

This notebook shows how to compute conditional probabilities on MDPs with Storm's Python bindings (`stormpy`).
The algorithms exposed here are the ones evaluated in the paper: `bisection`, `bisection_pt` (policy tracking),
`bisection_advanced`, `bisection_advanced_pt`, `restart`, and `policy_iteration`.

For the CLI-based benchmarks we run in the paper, see [benchmarks/rq1/](../benchmarks/rq1/).

## 1. Load an example MDP

We use the `wlan-BOFF=4` MDP (~2.1M states) as an example. It is based on the well known WLAN MDP model.

In [1]:
import stormpy as sp

MODEL_PATH = "../benchmarks/rq1/models/concrete-mdps/wlan-BOFF=4.drn"

model = sp.build_model_from_drn(MODEL_PATH)
print(
    f"type={model.model_type}  states={model.nr_states}  transitions={model.nr_transitions}"
)

type=ModelType.MDP  states=2138800  transitions=4566953


## 2. A conditional property

We ask for `Pmax(φ | ψ)` with `φ = F "collision8"` conditioned on `ψ = F "collision2"`.
In Storm's PRCTL syntax the conditional is written with `||` (read as “given”).

In [2]:
prop = sp.parse_properties('Pmax=? [F "collision8" || F "collision2"]')[0]
print(prop)

(1): Obtain the values of the '"init"'-states with values described by 'Pmax=? [(F "collision8") || (F "collision2")]'


## 3. Picking an algorithm via the `Environment` and using it

The conditional algorithm and its precision are stored in the model-checker environment:

```python
env.model_checker_environment.conditional.algorithm = sp.ConditionalAlgorithmSetting.<name>
env.model_checker_environment.conditional.precision = <rational>
```

Below we go over all methods and time them.

In [ ]:
import time

METHODS = [
    sp.ConditionalAlgorithmSetting.bisection,
    sp.ConditionalAlgorithmSetting.bisection_pt,
    sp.ConditionalAlgorithmSetting.bisection_advanced,
    sp.ConditionalAlgorithmSetting.bisection_advanced_pt,
    sp.ConditionalAlgorithmSetting.restart,
    sp.ConditionalAlgorithmSetting.policy_iteration,
]


def run(model, prop, algorithm):
    env = sp.Environment()
    env.model_checker_environment.conditional.algorithm = algorithm
    t0 = time.monotonic()
    result = sp.model_checking(model, prop, environment=env, only_initial_states=True)
    return result.at(model.initial_states[0]), time.monotonic() - t0


for m in METHODS:
    value, elapsed = run(model, prop, m)
    print(f"{m.name:<25s} value={float(value):.4e}  ({elapsed:.2f} s)")

bisection                 value=1.6211e-10  (0.41 s)
bisection_pt              value=1.6211e-10  (0.40 s)
WARN  (ConditionalHelper.cpp:1055): Precision of non-exact type reached during bisection method. Result might be inaccurate.
bisection_advanced        value=1.6211e-10  (0.41 s)
WARN  (ConditionalHelper.cpp:1055): Precision of non-exact type reached during bisection method. Result might be inaccurate.
bisection_advanced_pt     value=1.6211e-10  (0.39 s)
restart                   value=1.6211e-10  (0.40 s)
policy_iteration          value=1.6211e-10  (0.39 s)


## 4. Bounded (qualitative) queries

Instead of asking for the exact value we can ask whether it exceeds a threshold.

Treat is particularly cheap here since it only needs one iteration while not introducing extra loops in the model.

(The conditional probability on this model is tiny, so we pick a threshold near that scale.)

In [3]:
bounded = sp.parse_properties('Pmax>=1e-10 [F "collision8" || F "collision2"]')[0]

for m, name in [
    (sp.ConditionalAlgorithmSetting.bisection, "treat"),
    (sp.ConditionalAlgorithmSetting.restart, "restart"),
]:
    value, elapsed = run(model, bounded, m)
    print(f"{name:<12s} holds={value}  ({elapsed:.2f} s)")

NameError: name 'run' is not defined

## 5. Exact arithmetic

For sound, rational-number results we load the model with exact value types.
The result is a `pycarl.gmp.Rational` rather than a float.

In [ ]:
from stormpy import _core, _convert_sparse_model, _ValueType

exact_intermediate = _core._build_sparse_exact_model_from_drn(MODEL_PATH)
exact_model = _convert_sparse_model(exact_intermediate, value_type=_ValueType.EXACT)
print(f"is_exact={exact_model.is_exact}  states={exact_model.nr_states}")

value, elapsed = run(exact_model, prop, sp.ConditionalAlgorithmSetting.bisection)
print(f"exact bisection: {value}  ≈ {float(value):.10f}  ({elapsed*1000:.2f} ms)")

is_exact=True  states=2138800
exact bisection: 3135752941636705/19342813113834066795298816  ≈ 0.0000000002  (2290.54 ms)
